# 01 — Exploratory Data Analysis
**Dataset:** Kaggle Credit Card Fraud Detection  
**Goal:** Understand class imbalance, feature distributions, and correlations before modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['figure.facecolor'] = '#0f0c29'
plt.rcParams['axes.facecolor']   = '#0f0c29'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'

os.makedirs('../plots', exist_ok=True)
print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('Dtypes:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
df.describe()

## 2. Class Imbalance

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate   = df['Class'].mean()

print(f'Legitimate transactions : {class_counts[0]:,}')
print(f'Fraudulent transactions : {class_counts[1]:,}')
print(f'Fraud rate              : {fraud_rate:.4%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution', fontsize=14, color='white')

# Bar chart
colors = ['#3498db', '#e74c3c']
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Count', color='white')
axes[0].set_ylabel('Number of Transactions', color='white')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', color='white', fontweight='bold')

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=['Legitimate\n(99.83%)', 'Fraud\n(0.17%)'],
    colors=colors, autopct='%1.3f%%',
    textprops={'color': 'white'},
    startangle=90
)
axes[1].set_title('Proportion', color='white')

plt.tight_layout()
plt.savefig('../plots/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/class_imbalance.png')

## 3. Transaction Amount Distribution

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount: Fraud vs Legitimate', fontsize=13, color='white')

# KDE
legit['Amount'].clip(upper=1000).plot.kde(ax=axes[0], color='#3498db', label='Legitimate', linewidth=2)
fraud['Amount'].plot.kde(ax=axes[0], color='#e74c3c', label='Fraud', linewidth=2)
axes[0].set_title('KDE Plot (clipped at $1000)', color='white')
axes[0].set_xlabel('Amount ($)', color='white')
axes[0].legend()
axes[0].set_xlim(0, 1000)

# Box plot
df.boxplot(column='Amount', by='Class', ax=axes[1],
           boxprops=dict(color='white'),
           whiskerprops=dict(color='white'),
           medianprops=dict(color='#f39c12', linewidth=2),
           capprops=dict(color='white'),
           flierprops=dict(markerfacecolor='#e74c3c', markersize=2))
axes[1].set_title('Box Plot', color='white')
axes[1].set_xlabel('Class (0=Legit, 1=Fraud)', color='white')
axes[1].set_ylabel('Amount ($)', color='white')
plt.suptitle('')

print(f"Fraud   — Median: ${fraud['Amount'].median():.2f} | Mean: ${fraud['Amount'].mean():.2f} | Max: ${fraud['Amount'].max():.2f}")
print(f"Legit   — Median: ${legit['Amount'].median():.2f} | Mean: ${legit['Amount'].mean():.2f} | Max: ${legit['Amount'].max():.2f}")

plt.tight_layout()
plt.savefig('../plots/amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Time Distribution — When Does Fraud Occur?

In [ ]:
# Convert seconds to hours
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Time Distribution', fontsize=13, color='white')

df[df['Class'] == 0]['Hour'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#3498db', alpha=0.8)
axes[0].set_title('Legitimate — By Hour', color='white')
axes[0].set_xlabel('Hour of Day', color='white')
axes[0].set_ylabel('Count', color='white')

df[df['Class'] == 1]['Hour'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#e74c3c', alpha=0.8)
axes[1].set_title('Fraud — By Hour', color='white')
axes[1].set_xlabel('Hour of Day', color='white')
axes[1].set_ylabel('Count', color='white')

plt.tight_layout()
plt.savefig('../plots/time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation with Fraud Class

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in correlations]
correlations.plot(kind='barh', ax=ax, color=colors, edgecolor='none')
ax.set_title('Feature Correlation with Fraud Class', fontsize=13, color='white')
ax.set_xlabel('Pearson Correlation', color='white')
ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig('../plots/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most negatively correlated with fraud:')
print(correlations.head(5))
print('\nTop 5 most positively correlated with fraud:')
print(correlations.tail(5))

## 6. Correlation Heatmap (Top Features)

In [ ]:
# Select most correlated features
top_features = list(correlations.abs().sort_values(ascending=False).head(10).index) + ['Class']
corr_matrix  = df[top_features].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax,
    linewidths=0.5, linecolor='#222',
    annot_kws={'size': 9, 'color': 'white'}
)
ax.set_title('Correlation Heatmap — Top Features', fontsize=13, color='white')
plt.tight_layout()
plt.savefig('../plots/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. EDA Summary

| Finding | Value |
|---|---|
| Total transactions | 284,807 |
| Fraud cases | 492 (0.17%) |
| Missing values | 0 |
| Duplicate rows | ~0 |
| Amount range | $0.00 – $25,691.16 |
| Fraud median amount | ~$22 |
| Legit median amount | ~$22 |

**Key insights:**
- Extreme class imbalance → need SMOTE
- V14, V12, V10 are most negatively correlated with fraud
- V4, V11 are most positively correlated with fraud
- Amount alone is not a strong fraud indicator
- No missing values → no imputation needed
- Amount and Time need scaling (V1-V28 are already PCA-transformed)